# RAUSNet — training on Google Colab (free tier)

Trains or retrains the grading model with the **same commands as the local machine**.
Run the cells top to bottom (`Runtime > Run all` works).

**Safety rules (see `docs/03_colab_and_runners.md`)**
- Free tier only: never buy compute units; use your own Google account.
- Patient data only in your **private** Drive folder. Never print or display patient images here,
  and never save this notebook with outputs back to the repository.
- The repository is cloned read-only. Commit and push from your own computer, so commits keep the
  author rules (JB-GB only).
- Every epoch is saved to Drive. If the session disconnects, run all cells again and training
  resumes from the last epoch.
- The last cell releases the GPU so the session does not sit idle.

## 1. Settings — edit only this cell

In [ ]:
REPO_URL = "https://github.com/JB-GB/AI_for_Biomedical_Engineers.git"
BRANCH = "PIA_devl"
DRIVE_DIR = "/content/drive/MyDrive/rausnet"   # private folder: data/, outputs/, models/
DATA_MODE = "synthetic"      # "synthetic" (generated here) or "drive" (DRIVE_DIR/data, copied locally)
TASK = "train"               # "train" (one run) or "retrain" (registry + promotion; needs ADD_MANIFESTS)
CONFIG = "configs/train/baseline.yaml"
OVERRIDES = []               # e.g. ["train.epochs=5"]
ADD_MANIFESTS = []           # retrain only, paths relative to the data root, e.g. ["incoming/batch_002/manifest.csv"]
AUTO_STOP = True             # release the GPU at the end

## 2. Runtime check

In [ ]:
import subprocess, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU: Runtime > Change runtime type > T4 GPU (free). CPU also works, only slower.")

## 3. Code (read-only clone) and dependencies

In [ ]:
import os
REPO = "/content/rausnet"
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only -q
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO}
# Colab already ships torch/torchvision with CUDA; pip keeps them because they satisfy the pins.
!pip install -q -e "{REPO}[train,serve]" onnxscript  # serve/onnxscript: ONNX export after retrain
!git -C {REPO} log --oneline -1

## 4. Drive (results persist here) and data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)

DATA_ROOT = "/content/data"            # local disk: much faster than reading from Drive
os.environ["RAUSNET_DATA_DIR"] = DATA_ROOT
if DATA_MODE == "synthetic":
    if not os.path.exists(f"{DATA_ROOT}/synth/manifest.csv"):
        # deterministic: same seed -> same images and same manifest hash as on the local machine
        !rausnet-synth --out {DATA_ROOT}/synth --patients 200 --seed 0
elif DATA_MODE == "drive":
    !mkdir -p {DATA_ROOT} && rsync -a --info=stats1 "{DRIVE_DIR}/data/" {DATA_ROOT}/
else:
    raise ValueError(DATA_MODE)

## 5. Train or retrain

`train` resumes the newest unfinished run in `DRIVE_DIR/outputs` (one with `last.pt` but no
`model_card.json`), otherwise starts a new run. `retrain` keeps the model registry in
`DRIVE_DIR/models` and applies the promotion rule; a promoted model is calibrated and exported
to ONNX in the same step (`calibration.json`, `model.onnx`, `model.json` in its version folder).

In [ ]:
import glob
os.chdir(REPO)
if TASK == "train":
    pending = sorted(d for d in glob.glob(f"{DRIVE_DIR}/outputs/*/")
                     if os.path.exists(d + "last.pt") and not os.path.exists(d + "model_card.json"))
    if pending:
        print("resuming", pending[-1])
        !python -m rausnet.train --resume "{pending[-1]}"
    else:
        sets = " ".join([f"output_dir={DRIVE_DIR}/outputs", *OVERRIDES])
        !python -m rausnet.train --config {CONFIG} --set {sets}
elif TASK == "retrain":
    assert ADD_MANIFESTS, "set ADD_MANIFESTS"
    adds = " ".join(f"--add {m}" for m in ADD_MANIFESTS)
    sets = " ".join([f"models_dir={DRIVE_DIR}/models", f"output_dir={DRIVE_DIR}/outputs", *OVERRIDES])
    !python -m rausnet.retrain --config configs/retrain/default.yaml {adds} --set {sets}
else:
    raise ValueError(TASK)

## 6. Results (numbers only, no images)

In [ ]:
import json
runs = sorted(glob.glob(f"{DRIVE_DIR}/outputs/*/model_card.json"))
if runs:
    card = json.load(open(runs[-1]))
    print(runs[-1])
    print(json.dumps({k: card[k] for k in ("device", "torch", "best_epoch", "test_combined", "manifest_sha256")},
                     indent=2))
if os.path.exists(f"{DRIVE_DIR}/models/registry.json"):
    !python -m rausnet.registry list --models-dir {DRIVE_DIR}/models

## 7. Release the GPU

In [ ]:
if AUTO_STOP:
    from google.colab import runtime
    runtime.unassign()